# Chapter 3


## Summarizing long documents with Ollama (gemma4:e2b)


### 3.1 Goal: summarize a document larger than the context window
When a document is too large for a single prompt, a practical strategy is **MapReduce summarization**:
1. split the text into manageable chunks
2. summarize each chunk independently (map)
3. summarize the collection of chunk summaries (reduce)

In this notebook, we apply that flow to `Moby-Dick.txt` using `ChatOllama` with `gemma4:e2b`.


In [ ]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

### 3.1.1 Prepare dependencies and model
After loading the source text, we import LangChain components and initialize the local model.

Compared to the OpenAI version, this notebook uses `ChatOllama(model="gemma4:e2b")`, so no API key is required.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel


In [ ]:
!pip install langchain langchain_ollama


In [ ]:
llm = ChatOllama(model="gemma4:e2b")


### 3.1.2 Split stage
`TokenTextSplitter` divides the input into chunks (`chunk_size=3000`, `chunk_overlap=100`).
The overlap helps preserve continuity between adjacent chunks.

`RunnableLambda` wraps custom Python logic so it can be composed in an LCEL chain.


In [ ]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

### 3.1.3 Map stage
Each chunk is summarized independently with the same prompt template.
`RunnableParallel` + `.map()` allows chunk-level processing in parallel, then `StrOutputParser` returns plain text summaries.


In [ ]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

### 3.1.4 Reduce stage
The reduce chain joins all map summaries into one text block (`summaries`) and asks the model for a final consolidated summary.
This is where many local summaries are compressed into one global result.


In [ ]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

### 3.1.5 Combined MapReduce chain
Here we compose the full pipeline in LCEL:
`split -> map -> reduce`.

The pipe operator (`|`) passes each step output to the next step.


In [ ]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

### 3.1.6 Execute and inspect
`invoke()` runs the whole chain on the full document and returns the final summary string.

With a local Ollama model, runtime depends mainly on chunk count and model speed; for quick tests, you can shorten the input text.


In [ ]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [ ]:
print(summary)

## Summarizing across documents with Ollama (gemma4:e2b)


In [ ]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [ ]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate


In [ ]:
!pip install langchain langchain_ollama


In [ ]:
llm = ChatOllama(model="gemma4:e2b")


In [ ]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [ ]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [ ]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [ ]:
full_summary = refine_summary(all_docs)
print(full_summary)